[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_01/4_ai.ipynb)

# Day 1: AI and ML (easy)

**How to work:** Answer concept questions out loud in 60 to 90 seconds before opening the solution: definition, why it matters, one example, one trade-off.

**Today (27 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Turn a churn worry into an ML problem | easy | 5 |
| Q2 | Know the target before you model | easy | 5 |
| Q3 | The model that is too good | easy | 7 |
| Q4 | Which kind of ML problem is it? | easy | 4 |
| Q5 | Does the model beat a one-line rule? | easy | 6 |

**Handbook:** [AI and ML](https://mahsa-agz.github.io/ds-handbook/#ai), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- `telco`: IBM Telco churn, 7,043 customers, one row = one customer. `Churn` (Yes/No), `tenure` (months), `Contract`, `MonthlyCharges`, `TotalCharges` (text with blanks), 9 service columns, demographics.
- `bank`: UCI Bank Marketing, 41,188 phone calls of a Portuguese bank, one row = one call. Target `y` (did the client subscribe a term deposit). Client columns (`age`, `job`, ...), campaign columns (`contact`, `month`, `duration`, `campaign`, `pdays`, `previous`, `poutcome`), macro columns.
- `titanic`: 891 passengers, one row = one passenger. `survived` (0/1), `pclass`, `sex`, `age` (has NaN), `sibsp`, `parch`, `fare`, `embarked`, plus derived columns (`alive`, `who`, ...).

The setup cell loads every table into a pandas DataFrame (from your Drive folder, or downloads it). Everything runs on the free Colab CPU in seconds. Numbers in the solutions come from scikit-learn 1.8; other versions can differ in the last digit.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import io, zipfile, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 120)

SEABORN = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/"
TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
BANK_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

def load_telco():
    """IBM Telco churn, 7,043 customers. Target: Churn (Yes/No)."""
    return pd.read_csv(data_path("telco_churn.csv", TELCO_URL))

def load_seaborn(name):
    """titanic, penguins, diamonds, mpg, tips ... from the seaborn-data repo."""
    return pd.read_csv(data_path(name + ".csv", SEABORN + name + ".csv"))

def load_bank():
    """UCI Bank Marketing (bank-additional-full), 41,188 calls. Target: y (yes/no). The source is a zip in a zip."""
    path = data_path("bank_marketing.csv")
    if not os.path.exists(path):
        print("downloading bank_marketing (zip)")
        outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(BANK_URL).read()))
        inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
        with open(path, "wb") as f:
            f.write(inner.read("bank-additional/bank-additional-full.csv"))
    return pd.read_csv(path, sep=";")

def check(fn, cases, tol=1e-6):
    """Runs fn on each (args, expected) case; numbers and arrays are compared with a tolerance."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*args)
            good = got is not None and np.shape(got) == np.shape(want) and bool(
                np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), atol=tol))
        except Exception as e:
            got, good = "error: " + repr(e)[:120], False
        ok += good
        shown = " ".join(repr(got).split())
        shown = shown if len(shown) < 100 else shown[:97] + "..."
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}(...) -> {shown}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

telco = load_telco()
print("telco", telco.shape)
bank = load_bank()
print("bank", bank.shape)
titanic = load_seaborn("titanic")
print("titanic", titanic.shape)

---
### Q1. Turn a churn worry into an ML problem

*easy, about 5 min*

The head of a telecom company says: "Too many customers leave. Can ML help?"

Frame it as an ML problem in about 90 seconds. Cover: the decision the model supports, the unit of prediction, the label (with a time window), the features you may use, the metric, the baseline, and how you would know it worked.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: a vague business goal. Pattern: ML framing, start from the decision, not from the model.

</details>

<details><summary><b>Hint 2</b></summary>

1. Decision: who gets a retention offer this month. 2. Unit: one active customer at a snapshot date. 3. Label: churned within the next 30 days after the snapshot. 4. Features: only what is known at the snapshot. 5. Offline metric (PR AUC, recall in the top 10%) and the business metric (saved revenue). 6. Baseline: a simple rule. 7. A/B test the offer.

</details>

<details><summary><b>Solution</b></summary>

**Model answer (about 90 s):**

"First I would ask what decision the model drives. Here: each month, which customers get a retention offer, given a budget for, say, 5% of customers. So the unit is one active customer on a snapshot date, and the label is 'cancels within the next 30 days'. That is binary classification, and what we really need is a good ranking, because we act on the top of the list.

Features must be known at the snapshot date: tenure, contract type, monthly charges, services, support tickets, usage trends. Anything recorded after the snapshot, like a cancellation reason, is leakage.

Churn is the minority class, so accuracy is useless. Offline I would use PR AUC and recall or precision in the top 5%. The baseline is a rule such as 'month-to-month contract and tenure under 6 months'; the model must beat it.

Finally, a churn score is not the goal. Some customers leave whatever we do, and some stay anyway. I would run an A/B test: offer versus no offer among high-score customers, and measure retained revenue minus offer cost. Later, an uplift model can target the persuadable customers."

**Key trade-off:** predicting churn is not the same as predicting who the offer will save.

**Why:** Examiners test whether you start from the decision and the data available at prediction time. The label window, the leakage check, the baseline and the online test are the parts most candidates skip.

**Common mistakes:** Jumping to 'I would use XGBoost'. No time window in the label. Using accuracy. Forgetting that the offer has a cost and that a high churn score does not mean the offer works.

**Learn more:** [ai-ml-framing](https://mahsa-agz.github.io/ds-handbook/#ai-ml-framing)

</details>

---
### Q2. Know the target before you model

*easy, about 5 min*

Use the `telco` table. Before any model:

1. Print the number of customers and the churn rate (share of `Churn == "Yes"`, 3 decimals).
2. Print the accuracy of a model that always predicts "No churn".
3. Print the churn rate per `Contract` type.

Then say in one sentence why a model with 80% accuracy may be weak here.

In [ ]:
# telco is loaded. Your code here

<details><summary><b>Hint 1</b></summary>

Signal: a classification task you have not seen yet. Pattern: check the base rate and the trivial baseline first.

</details>

<details><summary><b>Hint 2</b></summary>

1. `y = (telco['Churn'] == 'Yes').astype(int)`. 2. The mean of y is the churn rate. 3. The always-No model is right for every non-churner, so its accuracy is `1 - churn rate`. 4. `groupby('Contract')` and the mean of the 0/1 target.

</details>

<details><summary><b>Solution</b></summary>

```python
y = (telco["Churn"] == "Yes").astype(int)
print("customers:", len(telco))                         # out: customers: 7043
print("churn rate:", round(y.mean(), 3))                # out: churn rate: 0.265
print("majority-class accuracy:", round(1 - y.mean(), 3))  # out: majority-class accuracy: 0.735

rate = telco.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean()).round(3)
for contract, r in rate.items():
    print(contract, r)
# out: Month-to-month 0.427
# out: One year 0.113
# out: Two year 0.028
```

**Why:** About 26.5% of customers churn, so predicting "No" for everyone already gives 73.5% accuracy. An 80% accurate model is only a little better than doing nothing. Contract type alone separates risk strongly (42.7% churn on month-to-month versus 2.8% on two-year), which gives a strong simple baseline rule.

**Common mistakes:** Reporting accuracy without the base rate. Taking the mean of the text column. Skipping the simple segment rates, which often give the best baseline and the first product insight.

**Learn more:** [ai-ml-framing](https://mahsa-agz.github.io/ds-handbook/#ai-ml-framing), [ai-classification-metrics](https://mahsa-agz.github.io/ds-handbook/#ai-classification-metrics)

</details>

---
### Q3. The model that is too good

*easy, about 7 min*

The marketing team wants to know **before a call** which clients are likely to subscribe, so the call center can call them first. A colleague trained a logistic regression on all columns of `bank` and reports a test ROC AUC of about 0.93.

1. Read the column list. Which column cannot be used for this decision, and why?
2. Show the effect: fit the same pipeline with all columns, without that column, and with that column alone. Print the three test AUCs (stratified 75/25 split, `random_state=0`).

Pipeline: scale numeric columns, one-hot encode text columns, `LogisticRegression(max_iter=1000)`.

*Source: [UCI Bank Marketing dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing)*

In [ ]:
# bank is loaded. Your code here

<details><summary><b>Hint 1</b></summary>

Signal: a suspiciously high score and a column that is only filled in after the event. Pattern: target leakage (feature not available at prediction time).

</details>

<details><summary><b>Hint 2</b></summary>

1. `duration` is the length of the call; it is known only after the call ends, and a 0-second call always means "no". 2. Write one function that builds the ColumnTransformer for a list of columns, fits on train and returns the test AUC. 3. Call it three times.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

y = (bank["y"] == "yes").astype(int)
X = bank.drop(columns="y")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

def test_auc(cols):
    cat = [c for c in cols if X[c].dtype == object]
    num = [c for c in cols if c not in cat]
    prep = ColumnTransformer([("num", StandardScaler(), num),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    model = make_pipeline(prep, LogisticRegression(max_iter=1000))
    model.fit(X_tr[cols], y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te[cols])[:, 1])

all_cols = list(X.columns)
honest = [c for c in all_cols if c != "duration"]
print("AUC with duration:   ", round(test_auc(all_cols), 3))  # out: AUC with duration:    0.934
print("AUC without duration:", round(test_auc(honest), 3))    # out: AUC without duration: 0.786
print("AUC duration only:   ", round(test_auc(["duration"]), 3))  # out: AUC duration only:    0.83
```

**Why:** `duration` is measured after the call, and long calls happen because the client is interested. It is a consequence of the outcome, not a cause known in advance. Alone it reaches 0.83 AUC, more than all the honest columns together, and it lifts the full model from 0.786 to 0.934. The honest number for a 'whom to call first' model is about 0.79. The UCI page itself says to drop `duration` for a realistic predictive model.

**Complexity:** Logistic regression on about 31,000 rows and about 60 one-hot columns: around a second.

**Common mistakes:** Trusting a high score without asking 'when is each column known?'. Removing the column only from the test set. Forgetting other subtle leaks (for example a column updated after the outcome, or IDs that encode time).

**Learn more:** [ai-ml-framing](https://mahsa-agz.github.io/ds-handbook/#ai-ml-framing), [ai-features-imbalance](https://mahsa-agz.github.io/ds-handbook/#ai-features-imbalance)

</details>

---
### Q4. Which kind of ML problem is it?

*easy, about 4 min*

For each request, name the ML task type, the label (if any) and one metric:

1. Estimate how many minutes a food delivery will take.
2. Flag card transactions that are fraud (about 0.1% are).
3. Split our 2 million shoppers into a few groups for the marketing team.
4. Order the 500 candidate videos for a user's feed.
5. Forecast next week's daily orders per city.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: different output types. Pattern: map the output (number, class, group, order, future value) to a task type.

</details>

<details><summary><b>Hint 2</b></summary>

Ask for each: is there a label? Is the output a number, a class, an ordering, or a future value of a time series? Then pick a metric that matches the cost of errors.

</details>

<details><summary><b>Solution</b></summary>

| # | Task type | Label | Metric |
|---|---|---|---|
| 1 | Regression | actual delivery minutes | MAE (minutes are easy to explain); also check the share of late predictions |
| 2 | Binary classification, very imbalanced (or anomaly detection if labels are scarce) | chargeback / confirmed fraud | PR AUC, recall at a fixed precision or at a review budget |
| 3 | Unsupervised clustering | none | silhouette as a check, but mainly whether the groups are stable and actionable |
| 4 | Ranking (learning to rank) | watch time, finish, like | NDCG@k or recall@k offline; watch time in an A/B test |
| 5 | Time-series forecasting (regression over time) | future orders | MAPE or MAE per city, validated with a time-based split |

**Say out loud:** the output type picks the family; the cost of each error picks the metric.

**Why:** Naming the task type correctly in the first seconds tells the examiner you know which losses, metrics and validation schemes apply. Note the traps: fraud needs imbalance-aware metrics, forecasts need a time split, and ranking is judged on the top of the list.

**Common mistakes:** Calling the feed a classification problem and using accuracy. Using a random split for the forecast. Treating clustering quality as a single number.

**Learn more:** [ai-ml-framing](https://mahsa-agz.github.io/ds-handbook/#ai-ml-framing), [ai-unsupervised](https://mahsa-agz.github.io/ds-handbook/#ai-unsupervised)

</details>

---
### Q5. Does the model beat a one-line rule?

*easy, about 6 min*

Use `titanic`. Split it 70/30 (stratified on `survived`, `random_state=1`). On the test part, print the accuracy of:

1. the majority class (everybody dies),
2. the rule "women survive, men do not",
3. a logistic regression on `sex` (as 0/1), `pclass`, `age` (fill missing with the train median) and `fare`.

What would you tell a PM who wants a model?

In [ ]:
# titanic is loaded. Your code here

<details><summary><b>Hint 1</b></summary>

Signal: 'is ML worth it?'. Pattern: compare against trivial and rule-based baselines on the same test set.

</details>

<details><summary><b>Hint 2</b></summary>

1. `train_test_split(df, test_size=0.3, stratify=df['survived'], random_state=1)`. 2. Baseline accuracy is the share of zeros in the test labels. 3. The rule is `sex == 'female'`. 4. Fit the median on train, apply it to both parts, fit `LogisticRegression(max_iter=1000)`.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

df = titanic[["survived", "sex", "pclass", "age", "fare"]].copy()
X_tr, X_te = train_test_split(df, test_size=0.3, stratify=df["survived"], random_state=1)

# 1. majority class: everybody dies
print("majority baseline:", round((X_te["survived"] == 0).mean(), 3))      # out: majority baseline: 0.616
# 2. one rule: women survive, men do not
rule = (X_te["sex"] == "female").astype(int)
print("rule 'women survive':", round((rule == X_te["survived"]).mean(), 3))  # out: rule 'women survive': 0.802

# 3. logistic regression; the age median comes from the TRAIN part only
med_age = X_tr["age"].median()
def features(d):
    return pd.DataFrame({"female": (d["sex"] == "female").astype(int), "pclass": d["pclass"],
                         "age": d["age"].fillna(med_age), "fare": d["fare"]})
lr = LogisticRegression(max_iter=1000).fit(features(X_tr), X_tr["survived"])
print("logistic regression:", round(lr.score(features(X_te), X_te["survived"]), 3))  # out: logistic regression: 0.772
```

**Why:** Everybody-dies gives 0.616, the one-line rule gives 0.802, and the model gives 0.772: the model is **worse** than a rule you can explain in three words. Sex carries most of the signal; the model adds noisy linear effects of age and fare and has no sex-by-class interaction, so it flips some correct rule predictions. With 268 test passengers, 3 points is about 8 people, so one split cannot rank them reliably; use cross-validation before deciding. For the PM: ship the rule as the baseline, and build a model only if it beats the rule clearly in CV and the gain is worth the maintenance.

**Complexity:** Tiny data: milliseconds.

**Common mistakes:** Filling missing age with the median of the full data (a small leak from test into train). Comparing models on different splits. Not reporting a baseline at all.

**Learn more:** [ai-ml-framing](https://mahsa-agz.github.io/ds-handbook/#ai-ml-framing), [ai-logistic-regression](https://mahsa-agz.github.io/ds-handbook/#ai-logistic-regression)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Done for today.** Finish with the [theory flashcards for day 1](https://mahsa-agz.github.io/ds-handbook/#day-1).